## Example: US States Data

Import pandas and numpy

In [1]:
import pandas as pd
import numpy as np

Data can be found at Dropbox shared folder under the folder "data"

Read the three datasets, 
```
state-population.csv
state-areas.csv
state-abbrevs.csv
```
using the Pandas ``read_csv`` function:

In [2]:

pop = pd.read_csv('data/state-population.csv')
areas = pd.read_csv('data/state-areas.csv')
abbrevs = pd.read_csv('data/state-abbrevs.csv')

We want to compute a relatively straightforward result: rank US states and territories by their 2010 population density.
We clearly have the data here to find this result, but we'll have to combine the datasets to do so.

Merge `pop` and `abbrevs` based on common columns, and save the result to new variable `merged`. Use `outer merge` to make sure no data is thrown away due to mismatched labels:

In [5]:
merged = pd.merge(
    pop, 
    abbrevs, 
    how='outer',
    left_on='state/region', 
    right_on='abbreviation'
)
merged = merged.drop('abbreviation', axis=1) # drop duplicate info
merged.head()

,state/region,ages,year,population,state
0,AK,total,1990,553290.0,Alaska
1,AK,under18,1990,177502.0,Alaska
2,AK,total,1992,588736.0,Alaska
3,AK,under18,1991,182180.0,Alaska
4,AK,under18,1992,184878.0,Alaska


Check which columns have nulls:

In [6]:
merged.isnull().any()

state/region    False
ages            False
year            False
population       True
state            True
dtype: bool

Some of the ``population`` values are null; filter to see which these are!

In [7]:
merged[merged['population'].isnull()].head()

,state/region,ages,year,population,state
1872,PR,under18,1990,NaN,NaN
1873,PR,total,1990,NaN,NaN
1874,PR,total,1991,NaN,NaN
1875,PR,under18,1991,NaN,NaN
1876,PR,total,1993,NaN,NaN


Some of the new `state` entries are also null. In the `state/region` column of the `merged` DataFrame, find the unique values that are null:

In [8]:
merged.loc[merged['state'].isnull(), 'state/region'].unique()

array(['PR', 'USA'], dtype=object)

1. In the `merged` DataFrame, find the rows where the `state/region` column is equal to 'PR' and change the `state` column to 'Puerto Rico'
2. In the `merged` DataFrame, find the rows where the `state/region` column is equal to 'USA' and change the `state` column to 'United States'

In [9]:
merged.loc[merged['state/region'] == 'PR', 'state'] = 'Puerto Rico'
merged.loc[merged['state/region'] == 'USA', 'state'] = 'United States'

Check again, if any columns have nulls in the `merged` DataFrame:

In [10]:
merged.isnull().any()

state/region    False
ages            False
year            False
population       True
state           False
dtype: bool

No more nulls in the `state` column: we're all set!

Left merge the resulting `merged` DataFrame with the `areas` DataFrame using `state` as the common column. Save the result to new variable `final`.

In [14]:
final = pd.merge(merged, areas, on='state', how='left')
final.head()

,state/region,ages,year,population,state,area (sq. mi)
0,AK,total,1990,553290.0,Alaska,656425.0
1,AK,under18,1990,177502.0,Alaska,656425.0
2,AK,total,1992,588736.0,Alaska,656425.0
3,AK,under18,1991,182180.0,Alaska,656425.0
4,AK,under18,1992,184878.0,Alaska,656425.0


Again, let's check if any columns have nulls:

In [15]:
final.isnull().any()

state/region     False
ages             False
year             False
population        True
state            False
area (sq. mi)     True
dtype: bool

There are nulls in the ``area`` column; which states have null area?

In [16]:
final['state'][final['area (sq. mi)'].isnull()].unique()

array(['United States'], dtype=object)

Drop the null rows from `final` (because the population density of the entire United States is not relevant to our current discussion):

In [17]:
final.dropna(inplace=True)
final.head()

,state/region,ages,year,population,state,area (sq. mi)
0,AK,total,1990,553290.0,Alaska,656425.0
1,AK,under18,1990,177502.0,Alaska,656425.0
2,AK,total,1992,588736.0,Alaska,656425.0
3,AK,under18,1991,182180.0,Alaska,656425.0
4,AK,under18,1992,184878.0,Alaska,656425.0


Now we have all the data we need. To answer the question of interest, let's first select the portion of the data corresponding with the year 2010, and the total population. Save it to new variable `data2010`.

In [18]:

data2010 = final.query("year == 2010 & ages == 'total'")
data2010.head()

,state/region,ages,year,population,state,area (sq. mi)
43,AK,total,2010,713868.0,Alaska,656425.0
51,AL,total,2010,4785570.0,Alabama,52423.0
141,AR,total,2010,2922280.0,Arkansas,53182.0
149,AZ,total,2010,6408790.0,Arizona,114006.0
197,CA,total,2010,37333601.0,California,163707.0


Set the index of `data2010` to `state`:

In [19]:

data2010.set_index('state', inplace=True)

Now let's compute the population density as `pd.Series` and display it in order.

In [20]:
density = data2010['population'] / data2010['area (sq. mi)']
density.sort_values(ascending=False, inplace=True)
density.head()

state
District of Columbia    8898.897059
Puerto Rico             1058.665149
New Jersey              1009.253268
Rhode Island             681.339159
Connecticut              645.600649
dtype: float64

The result is a ranking of US states, plus Washington, DC, and Puerto Rico, in order of their 2010 population density, in residents per square mile.
We can see that by far the densest region in this dataset is Washington, DC (i.e., the District of Columbia); among states, the densest is New Jersey.

Check the end of the list:

In [21]:
density.tail()

state
South Dakota    10.583512
North Dakota     9.537565
Montana          6.736171
Wyoming          5.768079
Alaska           1.087509
dtype: float64

We see that the least dense state, by far, is Alaska, averaging slightly over one resident per square mile.